# 🧠 GAC-RAG: Graph-Augmented Code RAG — Interactive Demo

> **Novel RAG mechanism** that uses dependency graph traversal for true multi-hop reasoning over codebases.

## Architecture Recap

```
Query
  │
  ▼
[Layer 1] Semantic Anchor   ← ChromaDB vector search
  │
  ▼
[Layer 2] Graph Expansion   ← Neo4j N-hop traversal  ★ The novel part
  │
  ▼
[Layer 3] LLM Reranker      ← Claude prunes irrelevant nodes
  │
  ▼
Final Answer
```

---

## Prerequisites

1. Neo4j running on `bolt://localhost:7687` (see README for Docker command)
2. `.env` file with `ANTHROPIC_API_KEY`
3. `pip install -r requirements.txt`

## 0. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv('../.env')

from src.indexer import Indexer
from src.graph_store import GraphStore
from src.vector_store import VectorStore
from src.retriever import Retriever
from src.reranker import Reranker
from src.assistant import CodeAssistant

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.syntax import Syntax
from rich import print as rprint

console = Console()
print('✅ Imports OK')

---
## 1. Index the Sample Repository

The `sample_repo/` directory contains a small Python codebase simulating a payment system:

```
sample_repo/
├── services/
│   ├── payment_service.py    ← PaymentService.process()
│   ├── account_service.py    ← AccountService.debit()
│   └── notification_service.py
├── models/
│   └── transaction.py        ← Transaction.save()
└── utils/
    ├── database.py           ← DatabasePool.get_connection() ← ROOT BUG
    ├── retry.py              ← RetryHandler.attempt()
    └── event_bus.py          ← EventBus.emit()
```

In [ ]:
# Initialize stores
graph_store = GraphStore()   # Neo4j
vector_store = VectorStore() # ChromaDB

# Index the sample repo
indexer = Indexer()
nodes, edges = indexer.index_repo('../sample_repo')

# Store in both backends
graph_store.clear()
vector_store.clear()

graph_store.add_nodes_batch(nodes)
graph_store.add_edges_batch(edges)
vector_store.add_nodes_batch(nodes)

print(f'\n📊 Index Stats:')
print(f'   Nodes in Neo4j:    {graph_store.node_count()}')
print(f'   Edges in Neo4j:    {graph_store.edge_count()}')
print(f'   Vectors in Chroma: {vector_store.count()}')

---
## 2. Visualize the Code Graph

Let's see what nodes and edges were extracted from the sample repo.

In [ ]:
# Display nodes by type
table = Table(title='Indexed Code Nodes', show_header=True)
table.add_column('Kind', style='cyan', width=10)
table.add_column('Name', style='green', width=30)
table.add_column('File', style='yellow', width=45)
table.add_column('Lines', style='white', width=10)

for node in sorted(nodes, key=lambda n: (n.kind, n.name)):
    table.add_row(
        node.kind,
        node.name,
        node.file_path.replace('../sample_repo/', ''),
        f'{node.start_line}-{node.end_line}'
    )

console.print(table)

In [ ]:
# Display edges
edge_table = Table(title='Dependency Edges', show_header=True)
edge_table.add_column('Source', style='green', width=40)
edge_table.add_column('Edge Type', style='magenta', width=15)
edge_table.add_column('Target', style='cyan', width=40)

for edge in edges:
    src_short = edge.source_id.split('::')[-1] if '::' in edge.source_id else edge.source_id
    tgt_short = edge.target_id.split('::')[-1] if '::' in edge.target_id else edge.target_id
    edge_table.add_row(src_short, f'──{edge.kind}──▶', tgt_short)

console.print(edge_table)

---
## 3. Layer 1 — Semantic Anchor (Vector Search Only)

This is what **standard RAG** does: find the top-k most similar nodes by cosine similarity.

In [ ]:
retriever = Retriever(graph_store, vector_store)

QUERY = "Why does payment processing fail silently when the database is down?"
print(f'Query: {QUERY}\n')

anchor_hits = retriever._layer1_semantic_anchor(QUERY)

print('🔎 Layer 1 — Top-5 Semantic Anchors (Standard RAG would stop here):')
print('-' * 60)
for node_id, score in anchor_hits:
    short_id = node_id.split('::')[-1] if '::' in node_id else node_id
    print(f'  score={score:.3f}  {short_id:<30} [{node_id}]')

---
## 4. Layer 2 — Graph Expansion (The Novel Part ★)

Starting from the anchor nodes, we **traverse the dependency graph** N hops.
Scores decay by `relevance_decay` per hop.

In [ ]:
anchor_ids = [node_id for node_id, _ in anchor_hits]
anchor_scores = {node_id: score for node_id, score in anchor_hits}

expanded_nodes = retriever._layer2_graph_expand(anchor_ids, anchor_scores)

print(f'🕸️  Layer 2 — Graph Expansion (max_hops={retriever.max_hops})')
print(f'   Standard RAG retrieved: {len(anchor_hits)} nodes')
print(f'   GAC-RAG retrieved:      {len(expanded_nodes)} nodes\n')

# Show expanded nodes by hop distance
expand_table = Table(title='Expanded Context Nodes', show_header=True)
expand_table.add_column('Hop', style='yellow', width=5)
expand_table.add_column('Score', style='cyan', width=8)
expand_table.add_column('Kind', style='magenta', width=10)
expand_table.add_column('Name', style='green', width=30)
expand_table.add_column('File', style='white', width=40)

seen_ids = set()
for node in sorted(expanded_nodes, key=lambda n: (n.hop, -n.score)):
    if node.id not in seen_ids:
        expand_table.add_row(
            str(node.hop),
            f'{node.score:.3f}',
            node.kind,
            node.name,
            node.file_path.replace('../sample_repo/', '')
        )
        seen_ids.add(node.id)

console.print(expand_table)

---
## 5. Layer 3 — LLM Reranker

Claude evaluates each candidate node and **prunes irrelevant ones** before final generation.

In [ ]:
reranker = Reranker()

final_nodes, reasoning = reranker.rerank(QUERY, expanded_nodes, verbose=True)

print(f'\n🤖 Layer 3 — LLM Reranker Results:')
print(f'   Input nodes:  {len(expanded_nodes)}')
print(f'   Output nodes: {len(final_nodes)}')
print(f'   Reasoning: {reasoning}\n')

rerank_table = Table(title='Final Context After Reranking', show_header=True)
rerank_table.add_column('Name', style='green', width=30)
rerank_table.add_column('Kind', style='magenta', width=10)
rerank_table.add_column('Score', style='cyan', width=8)
rerank_table.add_column('File', style='white', width=45)

for node in final_nodes:
    rerank_table.add_row(
        node.name,
        node.kind,
        f'{node.score:.3f}',
        node.file_path.replace('../sample_repo/', '')
    )

console.print(rerank_table)

---
## 6. Full GAC-RAG Answer

Now generate the final answer with the full multi-hop context.

In [ ]:
assistant = CodeAssistant(
    repo_path='../sample_repo',
    graph_store=graph_store,
    vector_store=vector_store,
)

answer = assistant.ask(QUERY, use_reranker=True, verbose=False)

console.print(Panel(
    answer,
    title=f'[bold green]GAC-RAG Answer[/bold green]',
    subtitle=f'Query: {QUERY[:60]}...',
    border_style='green',
    padding=(1, 2),
))

---
## 7. ⚡ Side-by-Side Comparison: Standard RAG vs GAC-RAG

The key benchmark: how much better is GAC-RAG for multi-hop questions?

In [ ]:
comparison = assistant.compare_with_naive_rag(QUERY)

print('=' * 80)
print('STANDARD RAG (Vector Search Only)')
print('=' * 80)
print(f"Nodes retrieved: {comparison['naive_rag']['nodes_retrieved']}")
print()
print(comparison['naive_rag']['answer'])

print()
print('=' * 80)
print('GAC-RAG (Graph-Augmented, 3-Layer)')
print('=' * 80)
print(comparison['gac_rag']['answer'])

---
## 8. 🔀 Query Decomposition Mode

For very complex multi-hop questions, GAC-RAG can **automatically decompose** the query into sub-questions, retrieve for each, then answer holistically.

In [ ]:
COMPLEX_QUERY = "Trace the complete data flow from a payment request to a user notification, and identify all points where a failure could propagate silently."

print(f'Complex Query: {COMPLEX_QUERY}\n')
print('Running query decomposition + multi-hop retrieval...\n')

answer_decomposed = assistant.ask_with_decomposition(COMPLEX_QUERY, verbose=True)

console.print(Panel(
    answer_decomposed,
    title='[bold blue]GAC-RAG with Query Decomposition[/bold blue]',
    border_style='blue',
    padding=(1, 2),
))

---
## 9. 🧪 Try Your Own Query

Run any question about the sample codebase!

In [ ]:
# ✏️ Edit this query!
MY_QUERY = "How does a refund get processed and what services are involved?"

answer = assistant.ask(MY_QUERY, use_reranker=True, verbose=True)

console.print(Panel(
    answer,
    title=f'[bold yellow]Answer[/bold yellow]',
    subtitle=MY_QUERY,
    border_style='yellow',
    padding=(1, 2),
))

---
## 10. 📊 Benchmarking: Multi-hop Question Set

Run GAC-RAG vs Standard RAG on a set of multi-hop questions and compare context richness.

In [ ]:
BENCHMARK_QUERIES = [
    "Why does payment processing fail silently when the database is down?",
    "How does a failed payment update the user's account status?",
    "What events are emitted during a successful payment?",
    "How does the retry mechanism interact with database connection failures?",
    "What is the complete call chain from PaymentService to EventBus?",
]

results = []
for q in BENCHMARK_QUERIES:
    anchor_hits = retriever._layer1_semantic_anchor(q)
    anchor_ids = [node_id for node_id, _ in anchor_hits]
    anchor_scores = {nid: s for nid, s in anchor_hits}
    expanded = retriever._layer2_graph_expand(anchor_ids, anchor_scores)
    
    results.append({
        'query': q[:55] + '...',
        'naive_nodes': len(anchor_hits),
        'gac_nodes': len(set(n.id for n in expanded)),
        'improvement': f'+{len(set(n.id for n in expanded)) - len(anchor_hits)} nodes'
    })

bench_table = Table(title='Benchmark: Context Richness (GAC-RAG vs Standard RAG)', show_header=True)
bench_table.add_column('Query', style='white', width=55)
bench_table.add_column('Naive\nNodes', style='red', width=8, justify='center')
bench_table.add_column('GAC-RAG\nNodes', style='green', width=10, justify='center')
bench_table.add_column('Improvement', style='cyan', width=15, justify='center')

for r in results:
    bench_table.add_row(
        r['query'],
        str(r['naive_nodes']),
        str(r['gac_nodes']),
        r['improvement']
    )

console.print(bench_table)
print(f"\n📈 On average, GAC-RAG retrieves {sum(r['gac_nodes'] for r in results)/len(results):.1f} nodes "
      f"vs {sum(r['naive_nodes'] for r in results)/len(results):.1f} for standard RAG")

---
## 11. Clean Up

In [ ]:
# Optional: clear the index
# graph_store.clear()
# vector_store.clear()

graph_store.close()
print('✅ Demo complete! Graph store connection closed.')

---
## Summary

| Layer | What it does | Backend |
|---|---|---|
| 1 — Semantic Anchor | Finds entry-point nodes by cosine similarity | ChromaDB |
| 2 — Graph Expansion ★ | Traverses call/import/inherit edges N hops | Neo4j |
| 3 — LLM Reranker | Prunes irrelevant nodes with Claude | Anthropic API |

**Key insight**: code is a graph, not a bag of text. Dependency-aware retrieval gives Claude the full picture for multi-hop reasoning.

📦 **GitHub**: [github.com/YOUR_USERNAME/gac-rag](https://github.com/YOUR_USERNAME/gac-rag)

⭐ Star the repo if this was useful!